# 07 — Improvements: Augmentation, Class Weighting & Ensemble

Three improvement strategies over the baseline:
1. **Data Augmentation** — time warping, amplitude scaling, Gaussian noise
2. **Oversampling** — `WeightedRandomSampler` to balance class frequencies
3. **Ensemble** — soft voting across all 4 trained models

We run an ablation study to quantify each contribution.

In [ ]:
import os, sys
import numpy as np
import torch
import torch.nn as nn
from torch.optim import Adam
from torch.optim.lr_scheduler import CosineAnnealingLR
import matplotlib.pyplot as plt
import pandas as pd

sys.path.insert(0, os.path.abspath('..'))
from src.dataset import get_dataloaders
from src.augment import AugmentedECGDataset, RandomAugment
from src.models import MODEL_REGISTRY
from src.evaluate import get_predictions, compute_metrics, CLASS_NAMES
from src.train import train_epoch, eval_epoch
from torch.utils.data import DataLoader

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

In [ ]:
X = np.load('../data/X.npy')
y = np.load('../data/y.npy')
_, _, test_loader, class_weights = get_dataloaders(X, y, batch_size=256)

## 1. Augmentation Ablation — ResNet-1D

In [ ]:
from sklearn.model_selection import train_test_split

idx = np.arange(len(y))
train_idx, temp_idx = train_test_split(idx, test_size=0.30, stratify=y, random_state=42)
val_idx, test_idx = train_test_split(temp_idx, test_size=0.50, stratify=y[temp_idx], random_state=42)

EPOCHS_ABL = 20  # short ablation
RESULTS_AUG = {}

aug_configs = [
    ('No Augmentation',  dict(augment=False)),
    ('+ Time Warp',      dict(p_warp=0.5, p_scale=0.0, p_noise=0.0, p_shift=0.0)),
    ('+ Amplitude Scale',dict(p_warp=0.0, p_scale=0.5, p_noise=0.0, p_shift=0.0)),
    ('+ Noise',          dict(p_warp=0.0, p_scale=0.0, p_noise=0.5, p_shift=0.0)),
    ('All Augmentations',dict(p_warp=0.5, p_scale=0.5, p_noise=0.5, p_shift=0.3)),
]

from src.dataset import ECGDataset
val_ds   = ECGDataset(X[val_idx],  y[val_idx])
test_ds  = ECGDataset(X[test_idx], y[test_idx])
val_loader_abl  = DataLoader(val_ds,  batch_size=256, shuffle=False)
test_loader_abl = DataLoader(test_ds, batch_size=256, shuffle=False)

for aug_name, aug_cfg in aug_configs:
    print(f'\n--- {aug_name} ---')
    if aug_cfg.get('augment') == False:
        train_ds = ECGDataset(X[train_idx], y[train_idx])
    else:
        train_ds = AugmentedECGDataset(X[train_idx], y[train_idx], augment=True, **aug_cfg)

    train_loader_abl = DataLoader(train_ds, batch_size=64, shuffle=True)

    model = MODEL_REGISTRY['resnet'](n_leads=2, n_classes=5).to(device)
    criterion = nn.CrossEntropyLoss(weight=class_weights.to(device))
    optimizer = Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
    scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS_ABL)

    best_va, best_state = 0, None
    for ep in range(1, EPOCHS_ABL+1):
        train_epoch(model, train_loader_abl, criterion, optimizer, device)
        _, va = eval_epoch(model, val_loader_abl, criterion, device)
        scheduler.step()
        if va > best_va:
            best_va = va
            best_state = {k: v.clone() for k, v in model.state_dict().items()}

    model.load_state_dict(best_state)
    yt, yp, ypr = get_predictions(model, test_loader_abl, device)
    m = compute_metrics(yt, yp, ypr)
    RESULTS_AUG[aug_name] = m
    print(f'  Test Acc: {m["accuracy"]*100:.2f}% | Macro F1: {m["macro_f1"]:.4f}')

In [ ]:
# Augmentation ablation table
rows = [{'Config': k, 'Accuracy (%)': round(v['accuracy']*100,2),
         'Macro F1': round(v['macro_f1'],4), 'Macro AUC': round(v['macro_auc'],4)}
        for k, v in RESULTS_AUG.items()]
df_aug = pd.DataFrame(rows).set_index('Config')
print('\n=== AUGMENTATION ABLATION ===')
print(df_aug.to_string())

## 2. Ensemble — Soft Voting

In [ ]:
MODEL_NAMES = ['cnn', 'resnet', 'lstm', 'transformer']
BASELINE_DIR = '../results/baseline'

loaded_models = []
for name in MODEL_NAMES:
    ckpt_path = f'{BASELINE_DIR}/{name}_best.pth'
    if not os.path.exists(ckpt_path):
        print(f'[skip] {name}')
        continue
    m = MODEL_REGISTRY[name](n_leads=2, n_classes=5).to(device)
    m.load_state_dict(torch.load(ckpt_path, map_location=device)['model_state'])
    m.eval()
    loaded_models.append(m)

print(f'Loaded {len(loaded_models)} models for ensemble')

In [ ]:
import torch.nn.functional as F

# Collect predictions from all models
all_probs, y_true_ens = [], None

for model in loaded_models:
    yt, _, ypr = get_predictions(model, test_loader, device)
    if y_true_ens is None:
        y_true_ens = yt
    all_probs.append(ypr)

# Soft voting: average probabilities
ensemble_probs = np.mean(all_probs, axis=0)
ensemble_preds = ensemble_probs.argmax(axis=1)

ens_metrics = compute_metrics(y_true_ens, ensemble_preds, ensemble_probs)
print(f'Ensemble Test Accuracy  : {ens_metrics["accuracy"]*100:.2f}%')
print(f'Ensemble Macro F1       : {ens_metrics["macro_f1"]:.4f}')
print(f'Ensemble Macro AUC      : {ens_metrics["macro_auc"]:.4f}')
print('\nPer-class F1:')
for name, f1 in zip(CLASS_NAMES, ens_metrics['per_class_f1']):
    print(f'  {name:20s}: {f1:.4f}')

In [ ]:
# Final summary: all models + ensemble
import json
try:
    with open('../results/summary_metrics.json') as f:
        summary = json.load(f)
except FileNotFoundError:
    summary = {}

summary['Ensemble'] = {
    'Accuracy (%)': round(ens_metrics['accuracy']*100, 2),
    'Macro F1': round(ens_metrics['macro_f1'], 4),
    'Weighted F1': round(ens_metrics['weighted_f1'], 4),
    'Macro AUC': round(ens_metrics['macro_auc'], 4),
}
with open('../results/summary_metrics.json', 'w') as f:
    json.dump(summary, f, indent=2)

df_final = pd.DataFrame(summary).T
print('\n=== FINAL RESULTS TABLE (including Ensemble) ===')
print(df_final.to_string())